# nb08 Feature Engineering


___

## Wat is feature engineering?

Feature engineering is het proces waarbij ruwe data wordt omgezet in informatieve inputvariabelen (*features*) die een machine learning-model in staat stellen relevante patronen te leren. Het gaat niet om het verzamelen van meer data, maar om het intelligenter representeren van bestaande data. In tijdreekscontexten — zoals parkeerpredicties — is het de meest bepalende stap voor modelkwaliteit, vaak belangrijker dan de keuze van het model zelf (Cerqueira et al., 2023). [sciencedirect](https://www.sciencedirect.com/science/article/pii/S2352146523012206)

**Academische formulering voor de thesis:**

> *"Feature engineering transformeert ruwe observaties in gestructureerde representaties die de onderliggende data-genererende processen blootleggen voor het leeralgoritme. In tijdreeksapplicaties omvat dit doorgaans vier categorieën: temporele structuurfeatures, lag-features, externe covariaten en ruimtelijke identiteitsfeatures (Wan et al., 2023; Cerqueira et al., 2023)."*

***

## De vier categorieën van features in nb08

### Categorie 1 — Temporele structuurfeatures (~8 features)

Dit zijn features die de **cyclische en kalenderstructuur** van de tijdreeks representeren. Parking-bezetting vertoont sterke dagelijkse en wekelijkse periodiciteit (H-T1, H-T2 bevestigd) en is daardoor bij uitstek geschikt voor cyclische encoding.

**Waarom sin/cos en niet gewoon het uurcijfer?**
Een ruwe uurfeature (`hour = 23`) en (`hour = 0`) liggen numeriek ver uiteen, terwijl ze gedragsmatig naast elkaar liggen (middernacht). Cyclische sin/cos-encoding lost dit op door de cirkelstructuur te respecteren (Wan et al., 2023): [mavmatrix.uta](https://mavmatrix.uta.edu/cgi/viewcontent.cgi?article=1010&context=utalibraries_openinitiativespubs)

$\text{hour\_sin} = \sin\!\left(\frac{2\pi \cdot h}{24}\right), \quad \text{hour\_cos} = \cos\!\left(\frac{2\pi \cdot h}{24}\right)$

| Feature | Periode | Literatuurbasis |
|---|---|---|
| `hour_sin`, `hour_cos` | 24u cyclus | Wan et al. (2023); Cerqueira et al. (2023) |
| `weekday_sin`, `weekday_cos` | 7-daagse cyclus | Fokker et al. (2021) |
| `month_sin`, `month_cos` | 12-maandse cyclus | H-T1 seizoenspatroon |
| `day_type_3` | Weekdag / zaterdag / zondag-feestdag | H-T1; Zhang et al. (2020) |
| `year_dummy` | Categorisch 2020=0, 2023/24=1 | H-T4 COVID-shift |

**Statistiek die nodig is:** geen fitting — louter wiskundige transformatie. Géén risk op leakage.

***

### Categorie 2 — Lag-features / autoregressive features (~3 features)

Dit zijn de **sterkste predictoren** in het model. Ze representeren de bezetting op eerdere tijdsmomenten en exploiteren de autocorrelatie die in H-T2 is aangetoond. De literatuur is eenduidig: historische bezetting is de meest informatieve predictor voor toekomstige bezetting (Lira et al., 2021; Yang et al., 2019). [ietresearch.onlinelibrary.wiley](https://ietresearch.onlinelibrary.wiley.com/doi/full/10.1049/itr2.12433)

| Feature | Lag | Motivatie |
|---|---|---|
| `occ_lag_1h` | t − 1 uur | Sterke positieve ACF bij lag-1 |
| `occ_lag_24h` | t − 24 uur | Dagelijkse periodiciteit (H-T2) |
| `occ_lag_168h` | t − 168 uur | Wekelijkse periodiciteit (H-T2) |

**Statistiek:** geen fitting — puur temporele verschuiving via `DataFrame.shift()`. Wel: expliciete NaN-check voor de eerste 168 uur van de holdout.

**⚠ Leakage-risico:** lag-features moeten worden aangemaakt *na* de train/holdout-split, op elk deel afzonderlijk. Dit is al geregeld via `train.parquet` en `holdout.parquet` uit nb07c.

***

### Categorie 3 — Externe covariaten (~10–12 features)

Dit zijn features die **externe invloeden** representeren: weer, kalender, events. Ze voegen signaal toe bovenop de structurele tijdreekspatronen.

**Weersfeatures:**

| Feature | Encoding | Motivatie | Statistiek nodig |
|---|---|---|---|
| `precip_bin` | 4 bins (droog/licht/matig/zwaar) | H-E1: drempelpatroon | Binning op trainingsdistributie |
| `wind_dummy` | 0/1 (>10 m/s) | H-E6: modal shift | Drempel hard-coded |
| `sun_duration_min` | Gestandaardiseerd | H-E7: laag prioriteit | Fit StandardScaler op train |
| ~~`temp_c`~~ | ~~Verwijderd~~ | Globale ρ = −0.013, Simpson's Paradox | — |

**Kalenderfeatures:**

| Feature | Encoding | Motivatie |
|---|---|---|
| `is_school_vacation` | Binair | H-E8: sterkste kalendereffect |
| `tier × is_school_vacation` | Interactieproduct | H-E8: Δr_rb = +0.117 |
| `is_national_holiday × tier` | Interactieproduct | H-E4: hertoetsen na Fischer z-fix |

**Event-features** (afgeleid via keyword-matching op `event_names_combined`):

| Feature | Encoding | Prioriteit |
|---|---|---|
| `is_kermis` | Binair | 🔴 Hoog — stadswijd medium effect |
| `is_voetbal × tier` | Interactieproduct | 🔴 Hoog — substitutie-effect |
| `is_festival × tier` | Interactieproduct | 🔴 Hoog — sterkste centrum-negatief |
| `is_carnaval × tier` | Interactieproduct | 🟡 Medium |
| `is_processie` | Binair | 🟡 Medium |
| `hours_to_next_event` | Continu (uren) | 🟡 Medium — cascade H-E3 |
| `hours_since_last_event` | Continu (uren) | 🟡 Medium — cascade H-E3 |

**Statistiek:** `StandardScaler` voor continue weersfeatures — **fit uitsluitend op trainingsset**, transform op beide.

***

### Categorie 4 — Ruimtelijke identiteitsfeatures (~4–12 features)

Dit zijn features die **welke parking** het is representeren. Ze zijn de dominantste ruimtelijke predictor (H-S1: η² klein maar RF-importance hoog).

| Feature | Encoding | Motivatie | Statistiek nodig |
|---|---|---|---|
| `parking_id` | Target encoding (regularized) | Dominante ruimtelijke predictor | Fit op train only — leakage-risico! |
| `tier_admin` | One-hot (2) | Administratieve tierindeling | Geen |
| `cluster_data` | Binair (0/1) | Data-gedreven k=2, silhouette=0.615 | Geen |
| `high_lt_pressure` | Binair | Ceiling effect P Hoogstraat/P Komet | Geen |

**Waarom target encoding voor `parking_id` en niet one-hot?**
Met 10 parkings is one-hot (10 kolommen) technisch haalbaar, maar target encoding is in de literatuur empirisch superieur voor lage-kardinaliteit categorische features in boomgebaseerde modellen (Pargent et al., 2021): het comprimeert de parkeerspelling naar één informatiedichte kolom en reduceert het risico op overfitting.  **Kritisch:** target encoding moet worden berekend met leave-one-out of k-fold cross-validation op de trainingsset om leakage te vermijden — scikit-learn's `TargetEncoder` doet dit automatisch. [arxiv](https://arxiv.org/abs/2104.00629v2)

***

## Totaaltelling: hoeveel features?

| Categorie | Aantal | Prioriteit |
|---|---|---|
| Temporele structuurfeatures | 8 | Allemaal 🔴 Hoog |
| Lag-features | 3 | Allemaal 🔴 Hoog |
| Externe covariaten (weer + kalender + events) | 12–14 | 🔴 Hoog tot 🟡 Medium |
| Ruimtelijke identiteitsfeatures | 4 | 🔴 Hoog |
| **Totaal ingaande feature-matrix** | **~27–29** | |
| Na VIF-pruning en importance-filter (verwacht) | **~22–25** | |

***

## Statistische stappen in nb08 — in volgorde

1. **Keyword-classificatie** `event_label_derived` op `event_names_combined` — geen statistiek, louter string-matching
2. **Lag-constructie** via `shift()` per `parking_id` — geen statistiek, NaN-check achteraf
3. **Cyclische transformatie** `sin/cos` — wiskundige formule, geen fitting
4. **Binning neerslag** `pd.cut()` met RMI-grenzen — fit op trainingsdistributie, apply op holdout
5. **StandardScaler** voor `wind_speed_ms`, `sun_duration_min` — fit op train, transform op beide
6. **Target encoding** `parking_id` — fit op train via k-fold, transform op beide (Pargent et al., 2021) [arxiv](https://arxiv.org/abs/2104.00629v2)
7. **Interactietermen** als rekenkundig product — geen statistiek
8. **VIF-herberekening** op volledige feature-matrix inclusief lag-features en interacties — multicollineariteitscheck
9. **Exporteren** als `train_features.parquet` en `holdout_features.parquet`

***

## Literatuursamenvatting voor de thesis-methodologie

> *"De feature engineering pipeline volgt vier categorieën die in de parking-predictieliteratuur empirisch zijn gevalideerd: (1) cyclische temporele encodings via sin/cos-transformatie voor dag-, week- en jaarcycli (Wan et al., 2023; Cerqueira et al., 2023), (2) autoregressive lag-features op tijdsafstanden van 1, 24 en 168 uur conform de empirisch vastgestelde ACF-structuur (Lira et al., 2021; Yang et al., 2019), (3) externe covariaten voor weer, kalender en event-cascade-effecten (Fokker et al., 2021; Oz, 2023), en (4) ruimtelijke identiteitsfeatures via geregulariseerde target encoding van parking_id (Pargent et al., 2021). Alle transformatieparameters worden uitsluitend geschat op de trainingsset (2020, 2023, 2024) en vervolgens toegepast op de holdoutset (2025) om temporele dataleakage te voorkomen (Cerqueira et al., 2023)."* [ietresearch.onlinelibrary.wiley](https://ietresearch.onlinelibrary.wiley.com/doi/full/10.1049/itr2.12433)

## De drie echte opties

**Optie A — Één model + tier als feature** *(huidige aanpak)*
- Voordelen: alle data samen, model leert tier-effecten via interacties, SHAP ontleedt achteraf
- Nadeel: minder interpreteerbaar voor beleidsmakers die "centrum vs. vesten" willen vergelijken

**Optie B — Aparte modellen op administratieve tier**
- Voordelen: eenvoudig, interpreteerbaar, voldoende data per model
- Nadeel: drie afwijkende parkings (Keerdok, Tinel, Hoogstraat) vervuilen beide modellen systematisch

**Optie C — Aparte modellen op data-gedreven cluster** *(methodologisch sterkst)*
- Cluster 0: P Grote Markt, P Kathedraal, P Lamot, P Veemarkt, P Keerdok, P Tinel → **6 parkings, ~78.000 rijen**
- Cluster 1: P Bruul, P Komet, P Maarten, P Hoogstraat → **4 parkings, ~52.000 rijen**
- Voordelen: gedragsmatig coherente groepen, silhouette = 0.615 valideert de split
- Nadeel: minder intuïtief voor beleidsmakers ("cluster 0" zegt niets zonder toelichting)

***

## Aanbevolen

**Optie A als hoofdmodel + Optie B als robustheidscheck.**

Concreet:
- Nb09–nb11: train één model met tier als feature (Optie A) — dit is je primaire resultaat
- Nb12 (SHAP): analyseer SHAP-waarden per administratieve tier — dit geeft de beleidsrelevante interpretatie
- Nb12 addendum: train ook twee aparte modellen per administratieve tier (Optie B) en vergelijk de MAPE — als de prestaties vergelijkbaar zijn, valideer je daarmee de tierstructuur; als Optie A beter scoort, heb je empirisch bewijs dat één geïntegreerd model de tierinteracties beter leert

**Academische formulering:**

> *"Hoewel een stratificatie naar administratieve tier (centrum vs. vesten/rand) beide deelmodellen voldoende data zou bieden (~65.000 respectievelijk ~57.000 trainingsrijen), toont de data-gedreven clusteranalyse (nb06) dat drie parkings gedragsmatig afwijken van hun administratieve label. Een gestratificeerde aanpak op administratieve basis introduceert bijgevolg systematische misclassificatie voor P Keerdok, P Tinel en P Hoogstraat. Als primaire modelarchitectuur wordt gekozen voor één geïntegreerd model met tier als expliciete feature en interactietermen; een gestratificeerd model per administratieve tier wordt als robustness check gerapporteerd in nb12."*



____

In [8]:
# ── Cel 01 · Imports & data laden ─────────────────────────────────────
from pathlib import Path
import pandas as pd
import numpy as np

ROOT      = Path("/Users/emilevandevoorde/Documents/mechelen_parking")
DATA_PROC = ROOT / "data_processed"

train   = pd.read_parquet(DATA_PROC / "train.parquet")
holdout = pd.read_parquet(DATA_PROC / "holdout.parquet")

# tier_admin aanmaken (conform nb07c)
for df in [train, holdout]:
    df["tier_admin"] = (
        df["parking_location_category"]
        .astype(str)
        .replace({"rand": "vesten_of_rand", "vesten": "vesten_of_rand"})
    )

print(f"Train  : {len(train):>7,} rijen × {train.shape[1]} kolommen")
print(f"Holdout: {len(holdout):>7,} rijen × {holdout.shape[1]} kolommen")


Train  : 129,837 rijen × 64 kolommen
Holdout:  87,600 rijen × 64 kolommen


## dtype audit

In [7]:
# ── Dtype audit: train vs. holdout ───────────────────────────────────
import pandas as pd

def dtype_audit(df, name):
    summary = pd.DataFrame({
        "dtype":    df.dtypes,
        "n_null":   df.isnull().sum(),
        "pct_null": (df.isnull().sum() / len(df) * 100).round(2),
        "n_unique": df.nunique(),
        "sample":   [df[c].dropna().iloc[0] if df[c].notna().any() 
                     else "ALL NULL" for c in df.columns]
    })
    print(f"\n{'='*60}")
    print(f"  {name}  —  {df.shape[0]:,} rijen × {df.shape[1]} kolommen")
    print(f"{'='*60}")
    print(summary.to_string())
    return summary

audit_train   = dtype_audit(train,   "TRAIN   (2020+2023+2024)")
audit_holdout = dtype_audit(holdout, "HOLDOUT (2025)")

# ── Vergelijk dtypes train vs. holdout ───────────────────────────────
print(f"\n{'='*60}")
print("  DTYPE MISMATCH train vs. holdout")
print(f"{'='*60}")
mismatch = (
    pd.DataFrame({
        "train_dtype":   audit_train["dtype"],
        "holdout_dtype": audit_holdout["dtype"]
    })
    .query("train_dtype != holdout_dtype")
)
if mismatch.empty:
    print("✅ Geen dtype-mismatches gevonden")
else:
    print(mismatch.to_string())

# ── Kolommen enkel in train of enkel in holdout ───────────────────────
only_train   = set(train.columns)   - set(holdout.columns)
only_holdout = set(holdout.columns) - set(train.columns)

print(f"\n{'='*60}")
print("  KOLOMMEN ENKEL IN TRAIN:")
print(only_train if only_train else "  ✅ Geen")
print("\n  KOLOMMEN ENKEL IN HOLDOUT:")
print(only_holdout if only_holdout else "  ✅ Geen")

# ── Specifieke checks voor nb08-relevante kolommen ────────────────────
print(f"\n{'='*60}")
print("  NB08-RELEVANTE KOLOMMEN — detail")
print(f"{'='*60}")
nb08_cols = [
    "rounded_hour", "occupancy_rate", "parking_id", "tier_admin",
    "parking_location_category", "year", "month", "hour", "weekday",
    "day_type_3", "temp_c", "precip_mm", "wind_speed_ms",
    "sun_duration_min", "is_event_day", "event_names_combined",
    "event_scale_max", "n_concurrent_events", "is_school_vacation",
    "is_national_holiday", "low_data_coverage", "system_blackout"
]
for col in nb08_cols:
    if col in train.columns:
        t_dtype  = train[col].dtype
        t_null   = train[col].isnull().sum()
        t_sample = train[col].dropna().iloc[0] if train[col].notna().any() else "NULL"
        print(f"  {col:<30} {str(t_dtype):<15} nulls={t_null:<6} sample={t_sample}")
    else:
        print(f"  {col:<30} ⚠ NIET AANWEZIG in train")



  TRAIN   (2020+2023+2024)  —  129,837 rijen × 64 kolommen
                                    dtype  n_null  pct_null  n_unique                                                            sample
parking_id                            str       0      0.00        10                                                           P Bruul
parking_id_hash                  category       0      0.00        10  0cdac8cfff7924a40f8ec805a661afd5663ee08c7ff707353430c982eb4b6d7b
parking_type                          str       0      0.00         1                                                       Car Parking
kind                                  str       0      0.00         1                                                         shortterm
year                                int32       0      0.00         3                                                              2020
month                               int32       0      0.00        12                                                       

# Notebook 08 — Feature Engineering Pipeline

**Project:** Tier-Stratified Occupancy Prediction and Scenario-Based Policy Simulation  
in Urban Parking Systems — A Case Study of Mechelen  
**Auteur:** Emile Van de Voorde  
**Input:** `data_processed/train.parquet` · `data_processed/holdout.parquet`  
**Output:** `data_processed/train_features.parquet` · `data_processed/holdout_features.parquet`

---

Deze notebook construeert de definitieve feature-matrix voor alle modelleer- en 
simulatienotebooks (nb09–nb13). Alle transformatieparameters worden uitsluitend 
geschat op de trainingsset (2020, 2023, 2024) en vervolgens toegepast op de 
holdoutset (2025) om temporele dataleakage te voorkomen (Cerqueira et al., 2023).

De feature-keuzes zijn empirisch onderbouwd door de 17 hypothesentoetsen uit 
FASE02 (nb05–nb07b) en bijgesteld via vier post-hoc analyses. Zie nb08-header 
markdown voor de volledige motivatie per feature.


### Voorbereidend
- `day_type_3` afleiden uit `weekday_int` + `is_national_holiday` (kolom bestaat nog niet)
- `tier_num` aanmaken als binaire integer van `tier_admin` (centrum=1, vesten=0) — nodig voor interacties

### Cyclische temporele encodings
- `hour_sin` + `hour_cos` uit `hour` (24u cyclus)
- `weekday_sin` + `weekday_cos` uit `weekday_int` (7-daagse cyclus)
- `month_sin` + `month_cos` uit `month` (12-maandse cyclus)

### Lag-features ⚠ leakage-kritisch
- Eerst sorteren op `parking_id` + `rounded_hour`
- `occ_lag_1h`, `occ_lag_24h`, `occ_lag_168h` via `groupby("parking_id").shift()`
- NaN-check na aanmaak — eerste 168 rijen per parking zijn verwacht NaN, droppen in nb09

### Weersfeatures ⚠ fit op train only
- `precip_bin`: 4 categorieën op vaste RMI-grenzen (0 / 2 / 5 mm) — geen fitting
- `wind_strong`: binaire drempel >10 m/s — geen fitting
- `sun_duration_scaled`: StandardScaler — **fit op train, transform op beide**
- `temp_c` **niet opnemen** — globale ρ = −0.013, Simpson's Paradox

### Ruimtelijke features ⚠ fit op train only
- `cluster_data`: hard-coded map uit nb06-resultaat (cluster 0 vs. 1)
- `high_lt_pressure`: binair — P Hoogstraat en P Komet = 1, rest = 0
- `parking_id_encoded`: TargetEncoder (cv=5) — **fit op train, transform op beide**

### Event-features
- Event-dummies **bestaan al**: `is_football_day`, `is_festival_day`, `is_kermis_day`, `is_carnival_day`, `is_procession_day`
- `is_event_day` generieke dummy **niet** opnemen als standalone feature — maskeert tegengestelde tier-effecten

### Interactietermen (rekenkundig product — geen fitting)
- `voetbal_x_centrum` = `is_football_day` × `tier_num`
- `festival_x_centrum` = `is_festival_day` × `tier_num`
- `carnaval_x_centrum` = `is_carnival_day` × `tier_num`
- `vakantie_x_centrum` = `is_school_vacation` × `tier_num`
- `feestdag_x_centrum` = `is_national_holiday` × `tier_num`

### Cascade-features
- `hours_to_next_event` en `hours_since_last_event` afleiden uit `is_event_day` per parking — forward/backward iteratie per groep

### Kalender
- `year_2020`: binair (2020=1, rest=0) — categorisch, **niet** ordinaal
- `is_school_vacation` en `is_national_holiday` bestaan al als float — casten naar int

### Kwaliteitscontroles
- VIF-check op volledige feature-matrix — alles >10 onderzoeken
- Dtype-check: alle features moeten float of int zijn vóór export — geen strings, geen categoricals
- Kolom-symmetrie bevestigen: train en holdout bevatten exact dezelfde feature-kolommen

### Export
- `train_features.parquet` en `holdout_features.parquet` opslaan naar `data_processed/`
- `rounded_hour`, `parking_id`, `tier_admin`, `year` mee exporteren als metadata — niet als modelfeature

___

## Stap 1 — Voorbereidende kolomconstructie

### 1.1 `day_type_3`

De bestaande `day_type`-kolom maakt enkel onderscheid tussen `'Weekday'` en 
`'Weekend'`, wat onvoldoende is voor het parkeergedrag in Mechelen. Uit H-T1 
(FASE02) bleek dat zaterdag structureel verschilt van zondag en nationale 
feestdagen: zaterdagen vertonen een commercieel retailprofiel met een 
middagpiek, terwijl zondagen en feestdagen een gedrukt, vlak profiel tonen 
vergelijkbaar met vakantiedagen (Zhang et al., 2020; Fokker et al., 2021).

De drie klassen zijn:
- `weekday` — maandag t.e.m. vrijdag, geen nationale feestdag
- `saturday` — zaterdag, geen nationale feestdag
- `sunday_holiday` — zondag of nationale feestdag (beide vertonen identiek gedragsprofiel)

### 1.2 `tier_num`

De string-kolom `tier_admin` wordt omgezet naar een binaire integer die nodig 
is als factor in de interactietermen (Stap 6). Zonder numerieke representatie 
kan het product `is_football_day × tier` niet worden berekend.


In [9]:
# ── Cel 02 · Voorbereidende kolomconstructie ──────────────────────────

for df in [train, holdout]:

    # ── day_type_3 ────────────────────────────────────────────────────
    df["day_type_3"] = df["weekday_int"].map({
        0: "weekday", 1: "weekday", 2: "weekday",
        3: "weekday", 4: "weekday",
        5: "saturday",
        6: "sunday_holiday"
    })
    # Nationale feestdag overschrijft altijd → sunday_holiday
    df.loc[df["is_national_holiday"] == 1, "day_type_3"] = "sunday_holiday"

    # ── tier_num ──────────────────────────────────────────────────────
    df["tier_num"] = (df["tier_admin"] == "centrum").astype(int)

# ── Verificatie ───────────────────────────────────────────────────────
print("day_type_3 verdeling (train):")
print(train["day_type_3"].value_counts())
print()
print("tier_num verdeling (train):")
print(train.groupby(["parking_id", "tier_num", "tier_admin"])
      .size().reset_index(name="n")
      .to_string(index=False))


day_type_3 verdeling (train):
day_type_3
weekday           90385
sunday_holiday    21928
saturday          17524
Name: count, dtype: int64

tier_num verdeling (train):
   parking_id  tier_num     tier_admin     n
      P Bruul         0 vesten_of_rand 12709
P Grote Markt         1        centrum 19044
 P Hoogstraat         1        centrum 12954
 P Kathedraal         1        centrum 12232
    P Keerdok         0 vesten_of_rand 16810
      P Komet         0 vesten_of_rand  3126
      P Lamot         1        centrum 14013
    P Maarten         0 vesten_of_rand  5944
      P Tinel         0 vesten_of_rand 18155
   P Veemarkt         1        centrum 14850


## Stap 2 — Cyclische temporele encodings

Parkeergedrag vertoont sterke periodieke structuren op drie tijdsschalen: 
dagelijks (ochtend-/avondspits), wekelijks (weekdag vs. weekend) en jaarlijks 
(seizoenen, schoolvakanties). Ruwe numerieke representatie van deze cycli 
(bijv. hour=23 en hour=0) introduceert een artificiële discontinuïteit aan 
de grens van elke cyclus, terwijl de twee tijdsmomenten gedragsmatig naast 
elkaar liggen.

De standaardoplossing in de tijdreeksliteratuur is sinusoïdale encoding 
(Wan et al., 2023; Cerqueira et al., 2023):

$$\text{feature\_sin} = \sin\!\left(\frac{2\pi \cdot t}{T}\right), \quad
\text{feature\_cos} = \cos\!\left(\frac{2\pi \cdot t}{T}\right)$$

waarbij $T$ de periode is (24 voor uur, 7 voor weekdag, 12 voor maand). 
Het sin/cos-paar samen beschrijft een unieke positie op de cirkel — 
één component alleen is onvoldoende omdat bijv. uur=6 en uur=18 
dezelfde sinuswaarde geven maar gedragsmatig totaal verschillen.

> **Geen fitting vereist** — louter wiskundige transformatie, identiek 
> toegepast op train en holdout. Geen leakage-risico.


In [10]:
# ── Cel 03 · Cyclische temporele encodings ────────────────────────────

for df in [train, holdout]:
    # Uur (T = 24)
    df["hour_sin"]    = np.sin(2 * np.pi * df["hour"] / 24)
    df["hour_cos"]    = np.cos(2 * np.pi * df["hour"] / 24)
    # Weekdag (T = 7)
    df["weekday_sin"] = np.sin(2 * np.pi * df["weekday_int"] / 7)
    df["weekday_cos"] = np.cos(2 * np.pi * df["weekday_int"] / 7)
    # Maand (T = 12)
    df["month_sin"]   = np.sin(2 * np.pi * df["month"] / 12)
    df["month_cos"]   = np.cos(2 * np.pi * df["month"] / 12)

# ── Verificatie: range moet [-1, 1] zijn, geen nulls ─────────────────
cyc_cols = ["hour_sin","hour_cos","weekday_sin",
            "weekday_cos","month_sin","month_cos"]

check = pd.DataFrame({
    "min":    [train[c].min()    for c in cyc_cols],
    "max":    [train[c].max()    for c in cyc_cols],
    "nulls":  [train[c].isnull().sum() for c in cyc_cols],
}, index=cyc_cols).round(6)

print("Cyclische encodings — range & null check (train):")
print(check.to_string())


Cyclische encodings — range & null check (train):
                  min       max  nulls
hour_sin    -1.000000  1.000000      0
hour_cos    -1.000000  1.000000      0
weekday_sin -0.974928  0.974928      0
weekday_cos -0.900969  1.000000      0
month_sin   -1.000000  1.000000      0
month_cos   -1.000000  1.000000      0


## Stap 3 — Lag-features (autoregressive)

Autoregressive lag-features zijn empirisch de sterkste predictoren in 
parkeer-occupancy modellen (Lira et al., 2021; Yang et al., 2019). 
De ACF-analyse in H-T2 (nb07) bevestigde drie significante pieken:

| Lag | Tijdsafstand | Gedragsmatige motivatie |
|-----|-------------|------------------------|
| 1   | 1 uur       | Sterke kortetermijn autocorrelatie |
| 24  | 24 uur      | Dagelijkse periodiciteit |
| 168 | 1 week      | Wekelijkse periodiciteit |

**Kritische vereisten:**
- Sorteren op `parking_id` + `rounded_hour` vóór shift — anders worden 
  waarden van parking A gebruikt als lag voor parking B
- Aanmaken ná de train/holdout-split — anders lekt holdout-informatie 
  achterwaarts in de trainingsset
- NaN bij de eerste 1/24/168 rijen per parking zijn **correct en verwacht** 
  — deze worden in nb09 gedropped vóór modeltraining


In [11]:
# ── Cel 04 · Lag-features ─────────────────────────────────────────────

# Sorteren is verplicht voor correcte shift per parking
train   = train.sort_values(
    ["parking_id", "rounded_hour"]).reset_index(drop=True)
holdout = holdout.sort_values(
    ["parking_id", "rounded_hour"]).reset_index(drop=True)

for df in [train, holdout]:
    grp = df.groupby("parking_id")["occupancy_rate"]
    df["occ_lag_1h"]   = grp.shift(1)
    df["occ_lag_24h"]  = grp.shift(24)
    df["occ_lag_168h"] = grp.shift(168)

# ── NaN-check ─────────────────────────────────────────────────────────
print("NaN-check lag-features:\n")
print(f"{'Feature':<16} {'Train nulls':>12} {'Holdout nulls':>14} "
      f"{'Verwacht (train)':>18}")
for lag, expected in [("occ_lag_1h",10),("occ_lag_24h",10),("occ_lag_168h",10)]:
    nt = train[lag].isnull().sum()
    nh = holdout[lag].isnull().sum()
    print(f"{lag:<16} {nt:>12} {nh:>14}      {expected} × 1 per parking")

print()
# Controleer dat nulls zich beperken tot de eerste rijen per parking
print("NaN-rijen occ_lag_168h in train — uitsluitend eerste rijen per parking?")
null_check = (
    train[train["occ_lag_168h"].isnull()]
    .groupby("parking_id")
    .agg(n_nulls=("rounded_hour","count"),
         eerste=("rounded_hour","min"),
         laatste=("rounded_hour","max"))
)
print(null_check.to_string())


NaN-check lag-features:

Feature           Train nulls  Holdout nulls   Verwacht (train)
occ_lag_1h                 10             10      10 × 1 per parking
occ_lag_24h               240            240      10 × 1 per parking
occ_lag_168h             1680           1680      10 × 1 per parking

NaN-rijen occ_lag_168h in train — uitsluitend eerste rijen per parking?
               n_nulls              eerste             laatste
parking_id                                                    
P Bruul            168 2020-01-01 00:00:00 2020-01-08 00:00:00
P Grote Markt      168 2020-01-01 00:00:00 2020-01-09 05:00:00
P Hoogstraat       168 2023-01-01 00:00:00 2023-01-11 07:00:00
P Kathedraal       168 2020-01-01 21:00:00 2020-08-23 03:00:00
P Keerdok          168 2023-01-17 00:00:00 2023-01-24 06:00:00
P Komet            168 2024-08-23 01:00:00 2024-08-30 17:00:00
P Lamot            168 2020-01-01 00:00:00 2020-01-09 10:00:00
P Maarten          168 2024-04-26 16:00:00 2024-05-03 15:00:00
P

## Stap 3b — Diagnose tijdreeksgaten in lag-features

Vóór we lag-features gebruiken in het model, moeten we begrijpen 
waar de NaN-waarden vandaan komen. Een `shift(168)` over een 
tijdreeksgat (bijv. parking start pas in augustus 2024) produceert 
een NaN die *niet* de eerste week van de reeks vertegenwoordigt, 
maar een structurele onderbreking. Zulke NaN-rijen mogen niet 
worden geïmputeerd — ze moeten worden gedropped in nb09, net zoals 
de initiële opstartNaN's.

Het onderscheid is methodologisch belangrijk: als we naïef imputeren,
injecteren we fictieve autocorrelatie in het model.


In [12]:
# ── Cel 04b · Diagnose tijdreeksgaten ────────────────────────────────

print("Startdatum per parking in train:")
start_check = (
    train.groupby("parking_id")["rounded_hour"]
    .agg(["min", "max", "count"])
    .rename(columns={"min":"start","max":"einde","count":"n_rijen"})
)
print(start_check.to_string())

print("\nVerwacht aaneengesloten uurdata per parking:")
for pid, row in start_check.iterrows():
    verwacht = int((row["einde"] - row["start"]).total_seconds() / 3600) + 1
    werkelijk = row["n_rijen"]
    gat = verwacht - werkelijk
    status = "✅" if gat == 0 else f"⚠ GAT van {gat} uur"
    print(f"  {pid:<15} verwacht={verwacht:>6} | werkelijk={werkelijk:>6} | {status}")


Startdatum per parking in train:
                            start               einde  n_rijen
parking_id                                                    
P Bruul       2020-01-01 00:00:00 2024-12-31 23:00:00    12709
P Grote Markt 2020-01-01 00:00:00 2024-12-31 23:00:00    19044
P Hoogstraat  2023-01-01 00:00:00 2024-12-31 23:00:00    12954
P Kathedraal  2020-01-01 21:00:00 2024-12-31 23:00:00    12232
P Keerdok     2023-01-17 00:00:00 2024-12-31 23:00:00    16810
P Komet       2024-08-23 01:00:00 2024-12-31 23:00:00     3126
P Lamot       2020-01-01 00:00:00 2024-12-31 23:00:00    14013
P Maarten     2024-04-26 16:00:00 2024-12-31 23:00:00     5944
P Tinel       2020-01-01 00:00:00 2024-12-31 23:00:00    18155
P Veemarkt    2020-01-01 00:00:00 2024-12-31 23:00:00    14850

Verwacht aaneengesloten uurdata per parking:
  P Bruul         verwacht= 43848 | werkelijk= 12709 | ⚠ GAT van 31139 uur
  P Grote Markt   verwacht= 43848 | werkelijk= 19044 | ⚠ GAT van 24804 uur
  P Hoogstraat 